<a href="https://colab.research.google.com/github/vituhaa/Multimodal-reasoning-for-STEM---edited/blob/main/%D0%98%D0%A1%D0%9F%D0%A0%D0%90%D0%92%D0%9B%D0%95%D0%9D%D0%9D%D0%AB%D0%99_%22Multimodal_Reasoning_for_STEM%22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

"HuggingFaceTB/SmolVLM-256M Instruct" model was selected to fine-tune.


In [ ]:
! nvidia-smi

In [1]:
! pip install -U bitsandbytes>=0.46.1

In [2]:
import torch
import requests
from transformers import AutoProcessor, AutoModelForImageTextToText
from transformers.image_utils import load_image
from peft import PeftModel

from datasets import load_dataset, concatenate_datasets
from transformers import BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import TrainingArguments, Trainer, tokenization_utils_base

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "HuggingFaceTB/SmolVLM-256M-Instruct"

In [ ]:
! pip install Levenshtein

In [4]:
import re
from Levenshtein import ratio

def delete_spaces(formula):
  formula = formula.strip()
  formula = re.sub(r'\s+', '', formula)
  return formula

def levenshtein_distance(pred, true):
  pred_normilized = delete_spaces(pred)
  true_normilized = delete_spaces(true)

  if pred_normilized == true_normilized:
    return 1.0

  distance = ratio(pred_normilized, true_normilized)
  return distance

In [ ]:
# train and test datasets:
train_raw = load_dataset("linxy/LaTeX_OCR", name="human_handwrite", split="train")
test_raw = load_dataset("linxy/LaTeX_OCR", name="human_handwrite", split="test")

Zero-shot inference

In [ ]:
def preprocess_dataset(example):
  image = example["image"]
  messages = [
      {
          "role": "user",
          "content": [
            {"type": "image"},
            {"type": "text", "text": "Convert this hand-written mathematical formula into LaTex format."}
          ]
      }
  ]
  prompt = processor.apply_chat_template(messages, add_generation_prompt=True)
  return {
      "images": [image],
      "texts": prompt
  }

processor = AutoProcessor.from_pretrained(model_name)
model = AutoModelForImageTextToText.from_pretrained(
    model_name,
    dtype=torch.bfloat16,
    _attn_implementation="eager"
).to(DEVICE)

In [ ]:
test_dataset = test_raw.map(preprocess_dataset)

In [9]:
lev_scores = []
for photo in test_dataset:
  inputs = processor(text=[photo['texts']], images=[photo['image']], return_tensors="pt")
  inputs = inputs.to(DEVICE)

  generated_ids = model.generate(**inputs, max_new_tokens=128)
  generated_texts = processor.batch_decode(
      generated_ids,
      skip_special_tokens=True,
  )
  pred_zero = generated_texts[0].strip()
  if "Assistant:" in pred_zero:
    pred_zero = pred_zero.split("Assistant:")[-1].strip()
  true = photo['text'].strip()
  lev_score = levenshtein_distance(pred_zero, true)
  lev_scores.append(lev_score)

print("Avg Levenstain Score Zero-Shot inference: ", sum(lev_scores) / len(lev_scores))

Avg Levenstain Score Zero-Shot inference:  0.8924195173348207


One-shot inference

In [10]:
inference_image_path = "/content/example.jpg"
inference_image = load_image(inference_image_path)

new_image_path = "/content/example_0.png"
new_image = load_image(new_image_path)

input_messages = [
    {
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": "Convert this hand-written mathematical formula into LaTex format."}
        ]
    },
    {
        "role": "assistant",
        "content": [
            {"type": "text", "text": r"\log z = \log r + i ( \theta + 2 n \pi )"}
        ]
    },
    {
        "role": "user",
        "content": [
            {"type": "image"},
            {"type": "text", "text": "Convert this hand-written mathematical formula into LaTex format."}
        ]
    }
]

prompt = processor.apply_chat_template(input_messages, add_generation_prompt=True)
inputs = processor(text=prompt, images=[inference_image, new_image], return_tensors="pt")
inputs = inputs.to(DEVICE)

generated_ids = model.generate(**inputs, max_new_tokens=128)
generated_texts = processor.batch_decode(
    generated_ids,
    skip_special_tokens=True,
)

print(*generated_texts)

User:

Convert this hand-written mathematical formula into LaTex format.
Assistant: \log z = \log r + i ( \theta + 2 n \pi )
User:

Convert this hand-written mathematical formula into LaTex format.
Assistant: \sqrt{x-y^{-2}+x^{2}+y^{2}+z^{2}}


In [10]:
lev_scores_one = []
for photo in test_dataset:
  prompt = processor.apply_chat_template(input_messages, add_generation_prompt=True)
  inputs = processor(text=prompt, images=[inference_image, photo['image']], return_tensors="pt")
  inputs = inputs.to(DEVICE)

  generated_ids = model.generate(**inputs, max_new_tokens=128)
  generated_texts = processor.batch_decode(
      generated_ids,
      skip_special_tokens=True,
  )
  pred_one = generated_texts[0].strip()
  if "Assistant:" in pred_one:
    pred_one = pred_one.split("Assistant:")[-1].strip()
  true = photo['text'].strip()
  lev_score_one = levenshtein_distance(pred_one, true)
  lev_scores_one.append(lev_score_one)
print("Avg Levenstain Score One-Shot inference: ", sum(lev_scores_one) / len(lev_scores_one))

Avg Levenstain Score One-Shot inference:  0.8929475063770358


Supervised fine-tuning (SFT) using linxy/LaTeX_OCR:train

In [ ]:
processor = AutoProcessor.from_pretrained(model_name)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForImageTextToText.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
    _attn_implementation="eager"
)

In [ ]:
model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=['down_proj','o_proj','k_proj','q_proj','gate_proj','up_proj','v_proj'],
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

In [10]:
def collate_fn(examples):
  texts = []
  images = []
  for example in examples:
      images.append(example['images'])
      texts.append(example['texts'])

  batch = processor(text=texts, images=images, return_tensors="pt", padding=True)
  labels = batch['input_ids'].clone()
  labels[labels == processor.tokenizer.pad_token_id] = -100
  image_token_id = processor.tokenizer.convert_tokens_to_ids("<|image|>")
  labels[labels == image_token_id] = -100
  batch["labels"] = labels

  return batch

https://huggingface.co/docs/transformers/v5.3.0/en/main_classes/trainer#transformers.TrainingArguments

In [11]:
def preprocess_train_dataset(example):
  image = example["image"]
  latex_output = example["text"]
  messages = [
      {
          "role": "user",
          "content": [
            {"type": "image"},
            {"type": "text", "text": "Convert this hand-written mathematical formula into LaTex format."}
          ]
      },
      {
          "role": "assistant",
          "content": [{"type": "text", "text": latex_output}]
      }
  ]
  prompt = processor.apply_chat_template(messages, add_generation_prompt=False) # для тренировки False
  return {
      "images": [image],
      "texts": prompt
  }

In [ ]:
train_dataset = train_raw.map(preprocess_train_dataset)

In [13]:
training_args = TrainingArguments(
    output_dir="./smolvlm_latex",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    num_train_epochs=1,
    learning_rate=2e-4,
    warmup_steps=50,
    optim="paged_adamw_8bit",
    weight_decay=0.01,
    gradient_accumulation_steps=4,
    fp16=True,
    bf16=False,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    remove_unused_columns=False,
    dataloader_num_workers=2
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    data_collator=collate_fn
)

In [17]:
print(len(train_dataset))

1200


In [14]:
trainer.train()
trainer.save_model("./smolvlm-fine-tuned")
processor.save_pretrained("./smolvlm-fine-tuned")

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss
100,0.046171,0.032060
200,0.027166,0.021778
300,0.019989,0.014138


['./smolvlm-fine-tuned/processor_config.json']

In [15]:
lev_scores_sft = []
for photo in test_dataset:
  # print(photo['image'])
  input_messages = [
      {
          "role": "user",
          "content": [
              {"type": "image"},
              {"type": "text", "text": "Convert this hand-written mathematical formula into LaTex format."}
          ]
      }
  ]
  prompt = processor.apply_chat_template(input_messages, add_generation_prompt=True)
  inputs = processor(text=prompt, images=[photo['image']], return_tensors="pt")
  inputs = inputs.to(DEVICE)

  generated_ids = model.generate(**inputs, max_new_tokens=128)
  generated_texts = processor.batch_decode(
      generated_ids,
      skip_special_tokens=True,
  )
  pred_sft = generated_texts[0].strip()
  if "Assistant:" in pred_sft:
    pred_sft = pred_sft.split("Assistant:")[-1].strip()
  # print(pred_sft)
  true = photo['text'].strip()
  lev_score_sft = levenshtein_distance(pred_sft, true)
  lev_scores_sft.append(lev_score_sft)
print("Avg Levenstain Score SFT inference: ", sum(lev_scores_sft) / len(lev_scores_sft))


Avg Levenstain Score SFT inference:  0.9158270074907277


In [ ]:
# сохраняю модель:
from huggingface_hub import notebook_login, HfApi

model_0 = AutoModelForImageTextToText.from_pretrained(model_name)
fn_model = PeftModel.from_pretrained(model_0, "vituuha/smolvlm-fine-tuned")
merged_model = fn_model.merge_and_unload()

merged_model.save_pretrained("./full_model_fn")
AutoProcessor.from_pretrained("vituuha/smolvlm-fine-tuned").save_pretrained("./full_model_fn")

notebook_login()
api = HfApi()
api.create_repo(repo_id="vituuha/smolvlm-full", exist_ok=True, private=False)
HfApi().upload_folder(folder_path="./full_model_fn", repo_id="vituuha/smolvlm-full", repo_type="model")

SFT using linxy/LaTeX_OCR:train + deepcopy/MathWriting-human

In [ ]:
train_num = 500

add_train_raw = load_dataset("deepcopy/MathWriting-human", split="train")
sub_train = add_train_raw.select(range(train_num))

In [ ]:
def preprocess_deepcopy_ds(example):
  image = example["image"]
  latex_output = example["latex"]
  messages = [
      {
          "role": "user",
          "content": [
            {"type": "image"},
            {"type": "text", "text": "Convert this hand-written mathematical formula into LaTex format."}
          ]
      },
      {
          "role": "assistant",
          "content": [{"type": "text", "text": latex_output}]
      }
  ]
  prompt = processor.apply_chat_template(messages, add_generation_prompt=False) # тренировка
  return {
      "images": [image],
      "texts": prompt
  }

add_train_dataset = sub_train.map(preprocess_deepcopy_ds)

In [18]:
combined_train = concatenate_datasets([train_dataset, add_train_dataset])

In [24]:
print(train_dataset[0])

{'image': <PIL.PngImagePlugin.PngImageFile image mode=RGBA size=515x71 at 0x7E3B49EC6180>, 'text': 'z _ { 1 } = r _ { 1 } ( \\cos \\theta _ { 1 } + i \\sin \\theta _ { 1 } )', 'images': [<PIL.PngImagePlugin.PngImageFile image mode=RGBA size=515x71 at 0x7E3B49EC7770>], 'texts': '<|im_start|>User:<image>Convert this hand-written mathematical formula into LaTex format.<end_of_utterance>\nAssistant: z _ { 1 } = r _ { 1 } ( \\cos \\theta _ { 1 } + i \\sin \\theta _ { 1 } )<end_of_utterance>\n'}


In [20]:
print(add_train_dataset[0])

{'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=187x141 at 0x7E3B4955C830>, 'latex': 'V(\\tilde{\\beta})', 'sample_id': '47f7bccab32dc3b3', 'split_tag': 'train', 'data_type': 'human', 'images': [<PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=187x141 at 0x7E3B4955C5C0>], 'texts': '<|im_start|>User:<image>Convert this hand-written mathematical formula into LaTex format.<end_of_utterance>\nAssistant: V(\\tilde{\\beta})<end_of_utterance>\n'}


In [21]:
add_train_dataset = add_train_dataset.remove_columns(['sample_id', 'split_tag', 'data_type'])

In [22]:
add_train_dataset = add_train_dataset.rename_column('latex', 'text')

In [23]:
print(add_train_dataset[0])

{'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=187x141 at 0x7E3B5A5A2CC0>, 'text': 'V(\\tilde{\\beta})', 'images': [<PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=187x141 at 0x7E3B49EE3710>], 'texts': '<|im_start|>User:<image>Convert this hand-written mathematical formula into LaTex format.<end_of_utterance>\nAssistant: V(\\tilde{\\beta})<end_of_utterance>\n'}


In [25]:
print(len(combined_train))

1700


In [26]:
training_args = TrainingArguments(
    output_dir="./smolvlm_latex_add",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=1,
    learning_rate=2e-4,
    warmup_steps=50,
    optim="paged_adamw_8bit",
    weight_decay=0.01,
    gradient_accumulation_steps=8,
    fp16=True,
    bf16=False,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    remove_unused_columns=False,
    dataloader_num_workers=2
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=combined_train,
    eval_dataset=test_dataset,
    data_collator=collate_fn
)


In [27]:
trainer.train()
trainer.save_model("./smolvlm-fine-tuned-add")
processor.save_pretrained("./smolvlm-fine-tuned-add")

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss
100,0.048546,0.040911


['./smolvlm-fine-tuned-add/processor_config.json']

In [40]:
ex_text = "Hello, what is your name?"
inputs = processor(text=ex_text, return_tensors="pt").to(DEVICE)
with torch.no_grad():
  output = model.generate(**inputs, max_new_tokens=128)
print(processor.decode(output[0], skip_special_tokens=True))

Hello, what is your name? What


In [41]:
lev_scores_sft_add = []
for photo in test_dataset:
  # print(photo['image'])
  input_messages = [
      {
          "role": "user",
          "content": [
              {"type": "image"},
              {"type": "text", "text": "Convert this hand-written mathematical formula into LaTex format."}
          ]
      }
  ]
  prompt = processor.apply_chat_template(input_messages, add_generation_prompt=True)
  inputs = processor(text=prompt, images=[photo['image']], return_tensors="pt")
  inputs = inputs.to(DEVICE)
  generated_ids = model.generate(**inputs, max_new_tokens=128)
  generated_texts = processor.batch_decode(
      generated_ids,
      skip_special_tokens=True,
  )

  answer = generated_texts[0].strip()
  print(answer)
#   if "Assistant:" in pred_sft_add:
#     pred_sft_add = pred_sft_add.split("Assistant:")[-1].strip()
#   # print(pred_sft)
#   true = photo['text'].strip()
#   lev_score_sft_add = levenshtein_distance(pred_sft_add, true)
#   lev_scores_sft_add.append(lev_score_sft_add)
# print("Avg Levenstain Score SFT-add inference: ", sum(lev_scores_sft_add) / len(lev_scores_sft_add))


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)


User:


Convert this hand-written mathematical formula into LaTex format.
Assistant: \


KeyboardInterrupt: 

In [ ]:
# сохраняю модель:
from huggingface_hub import login, HfApi

login()

model_0 = AutoModelForImageTextToText.from_pretrained(model_name)
fn_model = PeftModel.from_pretrained(model_0, "vituuha/smolvlm-fine-tuned")
merged_model = fn_model.merge_and_unload()

merged_model.save_pretrained("./smolvlm-fine-tuned-add", safe_serialization=False)
AutoProcessor.from_pretrained(model_name).save_pretrained("./smolvlm-fine-tuned-add")

api = HfApi()
api.create_repo(repo_id="vituuha/smolvlm-full-add", exist_ok=True, private=False)
HfApi().upload_folder(folder_path="./smolvlm-fine-tuned-add", repo_id="vituuha/smolvlm-full-add", repo_type="model")